# 03 — Real MIMIC Stage 1: Calibrated SAE Risk Forecaster

This notebook trains the supervised perception layer.

The RL controller must **not** learn perception from scratch.

Data split:
- TRAIN → fit classifier
- VALIDATION → fit probability calibration / choose thresholds
- TEST → final untouched evaluation


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

ROOT = Path.cwd().resolve().parents[1]
DATA = ROOT / "sae_trust_aware_v2" / "data"

print("Project root:", ROOT)
print("Data folder exists:", DATA.exists())

Project root: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning
Data folder exists: True


In [2]:
train = pd.read_parquet(DATA / "train_hourly.parquet")
val = pd.read_parquet(DATA / "val_hourly.parquet")
test = pd.read_parquet(DATA / "test_hourly.parquet")

BASE_FEATURES = [
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "gcs"
]

FEATURES = []

for c in BASE_FEATURES:
    FEATURES.extend([
        c,
        f"{c}_delta",
        f"{c}_mean3h"
    ])

print("TRAIN:", train.shape)
print("VALIDATION:", val.shape)
print("TEST:", test.shape)

print("\nFeatures:")
print(FEATURES)

TRAIN: (793, 30)
VALIDATION: (338, 30)
TEST: (453, 30)

Features:
['heart_rate', 'heart_rate_delta', 'heart_rate_mean3h', 'map', 'map_delta', 'map_mean3h', 'resp_rate', 'resp_rate_delta', 'resp_rate_mean3h', 'spo2', 'spo2_delta', 'spo2_mean3h', 'gcs', 'gcs_delta', 'gcs_mean3h']


In [3]:
# Remove rows with missing model inputs

train_m = train.dropna(
    subset=FEATURES + ["y"]
).copy()

val_m = val.dropna(
    subset=FEATURES + ["y"]
).copy()

test_m = test.dropna(
    subset=FEATURES + ["y"]
).copy()

X_train = train_m[FEATURES]
y_train = train_m["y"]

X_val = val_m[FEATURES]
y_val = val_m["y"]

X_test = test_m[FEATURES]
y_test = test_m["y"]

print("After missing-value filtering:")
print("TRAIN:", X_train.shape)
print("VALIDATION:", X_val.shape)
print("TEST:", X_test.shape)

print("\nPositive rates:")
print("TRAIN:", y_train.mean())
print("VALIDATION:", y_val.mean())
print("TEST:", y_test.mean())

After missing-value filtering:
TRAIN: (372, 15)
VALIDATION: (254, 15)
TEST: (379, 15)

Positive rates:
TRAIN: 0.08333333333333333
VALIDATION: 0.047244094488188976
TEST: 0.036939313984168866


In [4]:
clf = HistGradientBoostingClassifier(
    max_depth=3,
    max_iter=300,
    learning_rate=0.05,
    random_state=42
)

clf.fit(X_train, y_train)

raw_val = clf.predict_proba(X_val)[:, 1]
raw_test = clf.predict_proba(X_test)[:, 1]

print("Stage-1 classifier trained successfully.")

Stage-1 classifier trained successfully.


In [5]:
calibrator = IsotonicRegression(
    out_of_bounds="clip"
)

calibrator.fit(
    raw_val,
    y_val
)

risk_val = calibrator.transform(raw_val)
risk_test = calibrator.transform(raw_test)

print("Calibration completed.")
print("Validation risk range:",
      float(np.min(risk_val)),
      "to",
      float(np.max(risk_val)))

print("Test risk range:",
      float(np.min(risk_test)),
      "to",
      float(np.max(risk_test)))

Calibration completed.
Validation risk range: 0.0 to 0.05759162303664921
Test risk range: 0.0 to 0.05759162303664921


In [6]:
results = pd.DataFrame([
    {
        "split": "VALIDATION",
        "AUROC": roc_auc_score(y_val, risk_val),
        "AUPRC": average_precision_score(y_val, risk_val),
        "Brier": brier_score_loss(y_val, risk_val)
    },
    {
        "split": "TEST",
        "AUROC": roc_auc_score(y_test, risk_test),
        "AUPRC": average_precision_score(y_test, risk_test),
        "Brier": brier_score_loss(y_test, risk_test)
    }
])

display(results.round(4))

,split,AUROC,AUPRC,Brier
0,VALIDATION,0.5940,0.0576,0.0445
1,TEST,0.3967,0.0305,0.0366


In [7]:
val_out = val_m[
    [
        "subject_id",
        "stay_id",
        "hour",
        "sae_onset_hour",
        "y"
    ]
].copy()

val_out["risk"] = risk_val

test_out = test_m[
    [
        "subject_id",
        "stay_id",
        "hour",
        "sae_onset_hour",
        "y"
    ]
].copy()

test_out["risk"] = risk_test

val_out.to_parquet(
    DATA / "stage1_val_risk.parquet",
    index=False
)

test_out.to_parquet(
    DATA / "stage1_test_risk.parquet",
    index=False
)

print("Saved:")
print(DATA / "stage1_val_risk.parquet")
print(DATA / "stage1_test_risk.parquet")

Saved:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_trust_aware_v2/data/stage1_val_risk.parquet
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_trust_aware_v2/data/stage1_test_risk.parquet


In [8]:
print("Validation risk rows:", len(val_out))
print("Test risk rows:", len(test_out))

print("\nValidation columns:")
print(val_out.columns.tolist())

print("\nTest columns:")
print(test_out.columns.tolist())

print("\nStage 1 COMPLETE")

Validation risk rows: 254
Test risk rows: 379

Validation columns:
['subject_id', 'stay_id', 'hour', 'sae_onset_hour', 'y', 'risk']

Test columns:
['subject_id', 'stay_id', 'hour', 'sae_onset_hour', 'y', 'risk']

Stage 1 COMPLETE


### Required checks before moving to RL

Do not proceed if:
- test performance is being used to tune the forecaster,
- future rows leak into predictors,
- the same patient/stay appears in multiple splits,
- the SAE onset label is inconsistent,
- calibration is fit on training data instead of validation data.

Stage 2 consumes the frozen risk trajectory; it should not refit Stage 1.
